**AZURE DATABRICKS - FINAL ASSESSMENT**

**PART 1 — DATAFRAME (PYSPARK)**

In [0]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
df = spark.createDataFrame(data, columns)
df.show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [0]:
from pyspark.sql.functions import *
df = df.withColumn("test_cost", col("tests_count") * 2000)
df = df.withColumn("total_bill", col("consultation_fee") + col("test_cost"))
df = df.withColumn("patient_category",
    when(col("total_bill") > 15000, "High")
    .when(col("total_bill") > 8000, "Medium")
    .otherwise("Low")
)
df.show()

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|     2000|      7000|             Low|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|     4000|      7000|             Low|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|     2000|      3500|             Low|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     4000|      9000|          Medium|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|     2000|      9000|          Medium|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|     6000|      9000| 

In [0]:
df.filter(col("total_bill") > 10000).show()

+--------+------------+----+----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|city|department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+----+----------+----------------+-----------+---------+----------+----------------+
+--------+------------+----+----------+----------------+-----------+---------+----------+----------------+



In [0]:
df.groupBy("department","patient_category")\
.agg(count("visit_id").alias("total_patients"),
sum("total_bill").alias("total_revenue")).show()

+-----------+----------------+--------------+-------------+
| department|patient_category|total_patients|total_revenue|
+-----------+----------------+--------------+-------------+
| Cardiology|             Low|             2|        14000|
|Orthopedics|             Low|             1|         7000|
|Dermatology|             Low|             2|         9000|
| Cardiology|          Medium|             1|         9000|
|  Neurology|          Medium|             1|         9000|
|Orthopedics|          Medium|             1|         9000|
+-----------+----------------+--------------+-------------+



In [0]:
df.orderBy(col("total_bill").desc()).show()

+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|test_cost|total_bill|patient_category|
+--------+------------+---------+-----------+----------------+-----------+---------+----------+----------------+
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     4000|      9000|          Medium|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|     2000|      9000|          Medium|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|     6000|      9000|          Medium|
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|     2000|      7000|             Low|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|     4000|      7000|             Low|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|     2000|      7000| 

**PART 2 — SPARK SQL**

In [0]:
df.createOrReplaceTempView("patient_visits")

In [0]:
%sql
select * from patient_visits
where department = 'Dermatology';

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill,patient_category
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,3500,Low
108,Meera Iyer,Bangalore,Dermatology,1500,2,4000,5500,Low


In [0]:
%sql
select city, sum(total_bill) as total_revenue
from patient_visits
group by city;

city,total_revenue
Hyderabad,7000
Delhi,7000
Mumbai,3500
Bangalore,14500
Chennai,9000
Kolkata,9000
Ahmedabad,7000


In [0]:
%sql
select patient_name, total_bill
from patient_visits
order by total_bill desc
limit 3;

patient_name,total_bill
Priya Nair,9000
Vikram Singh,9000
Ananya Das,9000


In [0]:
%sql
select department, count(*) as total_patients
from patient_visits
group by department;

department,total_patients
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


**PART 3 — DELTA LAKE (CORE OPERATIONS)**

In [0]:
df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("patient_delta");

In [0]:
%sql
insert into patient_delta values
(109,'Santhosh','Tamil Nadu','Cardiology',5000,1,2000,7000,'Low');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
update patient_delta
set consultation_fee = 6000
where visit_id = 101;

num_affected_rows
1


In [0]:
%sql
delete from patient_delta
where city = 'Kolkata';

num_affected_rows
1


In [0]:
%sql
merge into patient_delta as target
using patient_visits as source
on target.visit_id = source.visit_id
when matched then
update set
  target.patient_name = source.patient_name,
  target.city = source.city,
  target.department = source.department,
  target.consultation_fee = source.consultation_fee,
  target.tests_count = source.tests_count,
  target.test_cost = source.test_cost,
  target.total_bill = source.total_bill,
  target.patient_category = source.patient_category
when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
8,7,0,1


**PART 4 — DELTA ADVANCED**

In [0]:
%sql
describe history patient_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-05-04T04:18:32.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2832783969736975),1b8ecf92-5a30-4a54-8c74-c7757c53d7ec,0504-035405-aetnqxjg-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 9, numRemovedBytes -> 23361, p25FileSize -> 3131, numDeletionVectorsRemoved -> 1, minFileSize -> 3131, numAddedFiles -> 1, maxFileSize -> 3131, p75FileSize -> 3131, p50FileSize -> 3131, numAddedBytes -> 3131)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T04:18:30.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#12854L = visit_id#11080L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2832783969736975),1b8ecf92-5a30-4a54-8c74-c7757c53d7ec,0504-035405-aetnqxjg-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 8, numTargetBytesAdded -> 20270, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 7, executionTimeMs -> 4635, materializeSourceTimeMs -> 395, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1611, numTargetRowsUpdated -> 7, numOutputRows -> 8, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2540)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T04:16:57.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2832783969736975),dd74f629-751a-4bb4-be35-d638c19a0265,0504-035405-aetnqxjg-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3136, p25FileSize -> 3091, numDeletionVectorsRemoved -> 1, minFileSize -> 3091, numAddedFiles -> 1, maxFileSize -> 3091, p75FileSize -> 3091, p50FileSize -> 3091, numAddedBytes -> 3091)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T04:16:54.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(city#12490 = Kolkata)""])",null,List(2832783969736975),dd74f629-751a-4bb4-be35-d638c19a0265,0504-035405-aetnqxjg-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1838, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1262, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 575)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T04:16:30.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2832783969736975),5ec1a658-6ab0-48de-9ef9-7dab6e4f15c3,0504-035405-aetnqxjg-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 8094, p25FileSize -> 3136, numDeletionVectorsRemoved -> 1, minFileSize -> 3136, numAddedFiles -> 1, maxFileSize -> 3136, p75FileSize -> 3136, p50FileSize -> 3136, numAddedBytes -> 3136)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-04T04:16:27.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#11844L = 101)""])",null,List(2832783969736975),5ec1a658-6ab0-48de-9ef9-7dab6e4f15c3,0

In [0]:
%sql
select * from patient_delta
version as of 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill,patient_category
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,2000,7000,Low
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,7000,Low
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,3500,Low
104,Priya Nair,Bangalore,Cardiology,5000,2,4000,9000,Medium
105,Vikram Singh,Chennai,Neurology,7000,1,2000,9000,Medium
106,Ananya Das,Kolkata,Orthopedics,3000,3,6000,9000,Medium
107,Karan Patel,Ahmedabad,Cardiology,5000,1,2000,7000,Low
108,Meera Iyer,Bangalore,Dermatology,1500,2,4000,5500,Low


**Explain the effect of VACUUM**
It removes old unused files and data from delta table to free up storage space

In [0]:
%sql
vacuum patient_delta retain 168 hours dry run;

path


**PART 5 — PARQUET → DELTA**

In [0]:
df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("patient_parquet_simulated")

In [0]:
%sql
create or replace table patient_delta
as select * from patient_parquet_simulated;

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

**PART 6 — INCREMENTAL LOAD**

In [0]:
df.write.format("delta")\
.mode("overwrite")\
.saveAsTable("patient_delta");

In [0]:
new_data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",6000,1,2000,8000,"Medium"),
(111,"Santhosh","Chennai","Neurology",7000,1,2000,9000,"Medium"),
(112,"Suresh","Mumbai","Orthopedics",3000,2,4000,7000,"Low")
]

columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count",
"test_cost",
"total_bill",
"patient_category"
]

new_df = spark.createDataFrame(new_data, columns)

new_df.createOrReplaceTempView("new_data")

In [0]:
%sql
merge into patient_delta as target
using new_data as source
on target.visit_id = source.visit_id

when matched then
update set
target.patient_name = source.patient_name,
target.city = source.city,
target.department = source.department,
target.consultation_fee = source.consultation_fee,
target.tests_count = source.tests_count,
target.test_cost = source.test_cost,
target.total_bill = source.total_bill,
target.patient_category = source.patient_category

when not matched then
insert *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,1,0,2


In [0]:
%sql
select * from patient_delta;

visit_id,patient_name,city,department,consultation_fee,tests_count,test_cost,total_bill,patient_category
102,Sneha Kapoor,Delhi,Orthopedics,3000,2,4000,7000,Low
103,Rahul Sharma,Mumbai,Dermatology,1500,1,2000,3500,Low
104,Priya Nair,Bangalore,Cardiology,5000,2,4000,9000,Medium
105,Vikram Singh,Chennai,Neurology,7000,1,2000,9000,Medium
106,Ananya Das,Kolkata,Orthopedics,3000,3,6000,9000,Medium
107,Karan Patel,Ahmedabad,Cardiology,5000,1,2000,7000,Low
108,Meera Iyer,Bangalore,Dermatology,1500,2,4000,5500,Low
101,Arjun Reddy,Hyderabad,Cardiology,6000,1,2000,8000,Medium
111,Santhosh,Chennai,Neurology,7000,1,2000,9000,Medium
112,Suresh,Mumbai,Orthopedics,3000,2,4000,7000,Low


**PART 9 — DATA GOVERNANCE**

In [0]:
%sql
show tables;

database,tableName,isTemporary
default,bronze_patient_visit,false
default,bronze_patient_visits,false
default,bronze_sales_inline,false
default,ecommerce_orders,false
default,gold_city_sales_summary,false
default,gold_hospital_summary,false
default,gold_patient_summary,false
default,patient_delta,false
default,patient_parquet,false
default,patient_parquet_simulated,false


In [0]:
%sql
show tables in default;

database,tableName,isTemporary
default,bronze_patient_visit,false
default,bronze_patient_visits,false
default,bronze_sales_inline,false
default,ecommerce_orders,false
default,gold_city_sales_summary,false
default,gold_hospital_summary,false
default,gold_patient_summary,false
default,patient_delta,false
default,patient_parquet,false
default,patient_parquet_simulated,false


In [0]:
%sql
create table high_value_patients as
select * from patient_delta
where total_bill > 8000;

num_affected_rows,num_inserted_rows


In [0]:
%sql
create table patient_records as
select * from patient_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql grant select on table patient_records 
to `azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com`;

In [0]:
%sql
select current_catalog(), current_schema();

current_catalog(),current_schema()
hexa_ws_7405609128804000,default


In [0]:
%sql describe history patient_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
11,2026-05-04T05:36:17.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2832783969736975),e12e2c67-50bf-4d6e-b092-f183b0a08f31,0504-035405-aetnqxjg-v2n,10,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 10660, p25FileSize -> 3126, numDeletionVectorsRemoved -> 1, minFileSize -> 3126, numAddedFiles -> 1, maxFileSize -> 3126, p75FileSize -> 3126, p50FileSize -> 3126, numAddedBytes -> 3126)",null,Databricks-Runtime/18.1.x-photon-scala2.13
10,2026-05-04T05:36:15.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#19525L = visit_id#19460L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2832783969736975),e12e2c67-50bf-4d6e-b092-f183b0a08f31,0504-035405-aetnqxjg-v2n,9,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 7580, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3093, materializeSourceTimeMs -> 114, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1253, numTargetRowsUpdated -> 1, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1691)",null,Databricks-Runtime/18.1.x-photon-scala2.13
9,2026-05-04T05:35:37.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2832783969736975),c9dbf1cf-4131-4e81-a755-a321f5a3d2e2,0504-035405-aetnqxjg-v2n,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3086, numDeletionVectorsRemoved -> 0, numOutputRows -> 8, numOutputBytes -> 3080)",null,Databricks-Runtime/18.1.x-photon-scala2.13
8,2026-05-04T04:50:38.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2832783969736975),680a78b0-57b6-463d-9a61-3abdce3f30fb,0504-035405-aetnqxjg-v2n,7,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3131, numDeletionVectorsRemoved -> 0, numOutputRows -> 8, numOutputBytes -> 3086)",null,Databricks-Runtime/18.1.x-photon-scala2.13
7,2026-05-04T04:18:32.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2832783969736975),1b8ecf92-5a30-4a54-8c74-c7757c53d7ec,0504-035405-aetnqxjg-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 9, numRemovedBytes -> 23361, p25FileSize -> 3131, numDeletionVectorsRemoved -> 1, minFileSize -> 3131, numAddedFiles -> 1, maxFileSize -> 3131, p75FileSize -> 3131, p50FileSize -> 3131, numAddedBytes -> 3131)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T04:18:30.000Z,144776549779146,azuser6412_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#12854L = visit_